# Structure à terme du skew : Rough Heston vs MPCE

Ce projet cherche à déterminer si un exposant de Hurst variable permet de reproduire, avec un seul facteur de volatilité, la structure à terme du skew observée sur le marché et son changement de vitesse de décroissance. La régularité de la volatilité est le mécanisme étudié ; la reproduction de la rupture est la question à résoudre.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
import visualisation as vis
config = vis.load_config()
ROOT = Path.cwd()


## Grille

Les maturités et les ratios K/F définissent les surfaces et la structure à terme du skew. K/F = 1 correspond à l’ATM. La grille est alignée sur les pas de simulation.

In [2]:
maturities = np.linspace(0.025, 1.0, 21)
strikes = np.unique(np.r_[np.linspace(0.85, 1.15, 11), 1.0])
config['grid'] = {'maturities': maturities.tolist(), 'strikes': strikes.tolist(), 'strike_coordinate': 'K/F'}
config['grid_alignment'] = 'nearest_step'  # ou 'strict'


## Rough Heston : exposant fixe

Les travaux de Bennedsen, Lunde et Pakkanen motivent une volatilité rugueuse. Rough Heston, associé aux travaux d’El Euch et Rosenbaum, conserve un exposant H constant. Le sous-jacent reste guidé par un brownien ; la variance porte la mémoire fractionnaire. Le pricing utilise Riccati et Fourier.

In [10]:
S0 = 100.0
r = 0.02
q = 0.0
config['market'] = {'spot': S0, 'rate': r, 'dividend': q}

H = 0.1
kappa_rh = 1.5
theta_rh = 0.04
xi_rh = 0.25
v0_rh = 0.04
rho_V_S = 0
config['rough_model'] = {'H': H, 'kappa': kappa_rh, 'theta': theta_rh, 'xi': xi_rh, 'v0': v0_rh, 'rho': rho_V_S}


## MPCE : exposant variable

Après avoir rendu aléatoire l’amplitude du bruit, on rend aléatoire sa régularité. Le cadre de Loboda, Mies et Steland guide cette construction. Le MPCE distingue amplitude et régularité, puis corrèle leurs innovations. H suit une diffusion de Jacobi ; les bornes configurées sont imposées numériquement. L’objectif est de reproduire la rupture du skew avec un seul facteur de volatilité à exposant variable. J'ajoute ici une correlation entre l'exposant de Hurst et les autres facteur

In [ ]:
kappa_mpce = 30
theta_mpce = 0.15
xi_mpce = 3
y0 = 0.2
V0 = 0.04
H0 = 0.1 
lambda_H = 20.0
eta_H = 0.1
zeta_H = 2
H_min = 0.03
H_max = 0.4

rho_H_y = -0.9
rho_H_S = 0.9
rho_y_S = -0.9
config['mpce_model'] = {'kappa': kappa_mpce, 'theta': theta_mpce, 'xi': xi_mpce, 'y0': y0, 'V0': V0,
    'H0': H0, 'lambda': lambda_H, 'eta': eta_H, 'zeta': zeta_H, 'H_min': H_min, 'H_max': H_max,
    'correlations': {'rho_H_y': rho_H_y, 'rho_H_S': rho_H_S, 'rho_y_S': rho_y_S}}
display(vis.correlation_matrix(config))


,H,y,S
H,1.0,-0.9,0.9
y,-0.9,1.0,-0.9
S,0.9,-0.9,1.0


## Historique

Le passé contribue à la log-variance et initialise y et H. Sa longueur et le nombre d’historiques sont deux choix distincts. Un historique donne un prix conditionnel ; plusieurs donnent une moyenne sur ces passés.

In [5]:
n_steps = 252
n_paths = 2**13
history_paths_percent = 1.0
history_length = 1.0
horizon = 1.0
seed = 1234
history_seed = 1234
config['simulation'].update(n_steps=n_steps, n_paths=n_paths, history_paths_percent=history_paths_percent,
    history_length=history_length, horizon=horizon, seed=seed, history_seed=history_seed)
config['output'].update(folder='resultats', overwrite=True, save_samples=True)
config['verification']['enabled'] = False  # tests ciblés inclus dans ce notebook
config['rough_numerical'].update(riccati_steps=192, fourier_upper=250.0, fourier_nodes=384)
config['simulation'].update(brownian_subcells=8, brownian_quadrature_order=64)


## Payoff et grecques

Les sensibilités du prix à H, à la vol-of-vol, au retour à la moyenne et au niveau long terme sont calculées par différences finies avec pricing Fourier. Le payoff et les pas sont modifiables. La sensibilité à la vol-of-vol est distincte du volga et peut être négative.

In [ ]:
payoff = {'type': 'call', 'notional': 1.0}  # ou 'put'
# payoff = {'type':'portfolio','legs':[{'type':'call','weight':1.,'strike_multiplier':1.},{'type':'call','weight':-1.,'strike_multiplier':1.1}]}
greek_steps = {'H': 0.001, 'xi': 0.001, 'kappa': 0.005, 'theta': 0.0001}
config['rough_greeks'] = {'enabled':True, 'payoff':payoff, 'steps':greek_steps, 'fd_abs_tolerance':1e-4, 'fd_rel_tolerance':0.01}

## Calcul Rough Heston

Ce bloc calcule les grecques sans lancer le MPCE. Dans les résultats fournis, les sensibilités aux paramètres de variance s’atténuent globalement vers l’échéance ; celle à H est plus importante à l’ATM à courte maturité.

In [11]:
import rough_heston as rh
engine_config = vis.to_engine_config(config)
greeks_rh = rh.parameter_greeks_fourier(config['rough_model'], config['rough_numerical'], config['market'], engine_config['grid'], config['rough_greeks'], payoff)
display(greeks_rh)
greek_folder = ROOT / 'resultats_rough_heston'
greek_folder.mkdir(exist_ok=True)
greeks_rh.to_csv(greek_folder/'greeks_rough_heston.csv',index=False)
greek_figures = vis.create_rough_greek_figures(greeks_rh, greek_folder)
for figure in greek_figures.values(): display(figure)

,T,moneyness,K,F,parameter,parameter_value,discounted_price,greek,greek_coarse,bump,bump_fine,linear_pnl_for_bump,fd_error_estimate,fd_tolerance,fd_stable,stencil,pricing_method,units,payoff_type
0,0.02381,0.85,85.040486,100.047630,H,0.10,15.000070,-0.001371,-0.001371,0.0010,0.00050,-0.000001,1.608550e-08,0.000114,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
1,0.02381,0.88,88.041915,100.047630,H,0.10,12.000658,-0.008279,-0.008279,0.0010,0.00050,-0.000008,3.775498e-08,0.000183,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
2,0.02381,0.91,91.043344,100.047630,H,0.10,9.005538,-0.037395,-0.037395,0.0010,0.00050,-0.000037,3.993280e-08,0.000474,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
3,0.02381,0.94,94.044773,100.047630,H,0.10,6.041291,-0.097749,-0.097749,0.0010,0.00050,-0.000098,3.191329e-08,0.001077,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
4,0.02381,0.97,97.046201,100.047630,H,0.10,3.259368,0.003818,0.003818,0.0010,0.00050,0.000004,4.079507e-08,0.000138,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
919,1.00000,1.03,105.080738,102.020134,theta,0.04,6.480903,54.798545,54.798555,0.0001,0.00005,0.005480,3.206252e-06,0.548085,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
920,1.00000,1.06,108.141342,102.020134,theta,0.04,5.373191,53.522488,53.522496,0.0001,0.00005,0.005352,2.543296e-06,0.535325,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
921,1.00000,1.09,111.201946,102.020134,theta,0.04,4.431937,51.112781,51.112786,0.0001,0.00005,0.005111,1.602293e-06,0.511228,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call
922,1.00000,1.12,114.262550,102.020134,theta,0.04,3.640137,47.850412,47.850413,0.0001,0.00005,0.004785,5.789265e-07,0.478604,True,central_2,Fourier_Lewis_Riccati,discounted_price_per_absolute_parameter_unit,call


## Comparaison des modèles

La comparaison examine si faire varier H permet de reproduire la structure à terme du skew de marché avec un seul facteur de volatilité. Les résultats du précédent essai réussi sont remplacés.

In [8]:
engine_config = vis.to_engine_config(config)
display(pd.DataFrame({'maturite_demandee': config['grid']['maturities'], 'maturite_effective': engine_config['grid']['maturities']}))
result = vis.run(config, ROOT)
display(result['surface'].query('requested_surface_point')[['model','T','moneyness','black_iv','price_se']])


,maturite_demandee,maturite_effective
0,0.02500,0.023810
1,0.07375,0.075397
2,0.12250,0.123016
3,0.17125,0.170635
4,0.22000,0.218254
5,0.26875,0.269841
6,0.31750,0.317460
7,0.36625,0.365079
8,0.41500,0.416667
9,0.46375,0.464286


,model,T,moneyness,black_iv,price_se
0,rough_heston,0.02381,0.85,0.317812,NaN
1,rough_heston,0.02381,0.88,0.293889,NaN
2,rough_heston,0.02381,0.91,0.268863,NaN
3,rough_heston,0.02381,0.94,0.242657,NaN
4,rough_heston,0.02381,0.97,0.215776,NaN
...,...,...,...,...,...
709,mpce,1.00000,1.03,0.273376,0.104647
710,mpce,1.00000,1.06,0.272484,0.102492
711,mpce,1.00000,1.09,0.271910,0.100438
712,mpce,1.00000,1.12,0.271634,0.098429


## Surfaces et skew

Les surfaces et les coupes montrent les sourires. La structure à terme du skew compare leurs pentes à l’ATM selon la maturité. L’ajustement en log-log permet de rechercher un changement de régime. Les deux axes des coupes servent à comparer les formes.

In [9]:
display(result['figures']['surfaces_interactives'])
display(result['figures']['skew_atm'])
display(result['figures']['skew_atm_loglog'])
display(result['verification'])


{'status': 'PARTIALLY_VERIFIED',
 'skew_consistency_failures': 0,
 'invalid_surface_points': 0,
 'limitations': ['Scenarios non calibres',
  'Conditionnement : voir simulation_metadata.json ; erreurs IID non valides entre historiques',
  'Convergence temporelle non etablie',
  'Ensemble historique : erreurs IID de trajectoires descriptives seulement',
  'BIC descriptif, erreurs correlees',
  'Coefficient corrige : anciens chiffres de diffusion invalides']}

## Vérification

Les contrôles concernent Rough Heston et la représentation des résultats. Ils distinguent la stabilité des différences finies de la précision du pricing.

In [ ]:
from pathlib import Path
import sys,json,hashlib,zipfile,ast,copy,time
import numpy as np,pandas as pd
from scipy.special import ndtr
P=ROOT;sys.path.insert(0,str(P))
import rough_heston as rh,visualisation as vis
c=vis.load_config();e=vis.to_engine_config(c);O=P/'resultats_rough_heston';O.mkdir(exist_ok=True)
report={'status':'PARTIALLY_VERIFIED','mpce_executed':False,'checks':{},'numerical_evidence':{}}
def record(name,value):report['checks'][name]=value;print(name,value,flush=True)
t=time.perf_counter()
g=rh.parameter_greeks_fourier(c['rough_model'],c['rough_numerical'],c['market'],e['grid'],c['rough_greeks'],c['rough_greeks']['payoff'])
g.to_csv(O/'greeks_rough_heston.csv',index=False)
assert len(g)==len(e['grid']['maturities'])*len(e['grid']['moneyness'])*4
assert np.isfinite(g.select_dtypes('number')).all().all();record('full_grid_count',len(g));record('fd_stable_count',int(g.fd_stable.sum()))
figs=vis.create_rough_greek_figures(g,O)
for parameter,figure in figs.items():
 assert len(figure.data)==len(e['grid']['maturities'])
 for trace,(T,cut) in zip(figure.data,g[g.parameter==parameter].groupby('T',sort=True)):
  cut=cut.sort_values('moneyness')
  np.testing.assert_array_equal(trace.x,cut.moneyness);np.testing.assert_array_equal(trace.y,cut.greek)
record('greek_plot_points_match_csv',True)
# Put-call parity and portfolio linearity, using the new Fourier price API.
ratios=np.array([.85,1.,1.15]);T=.5;F=100*np.exp(.02*T);disc=np.exp(-.02*T)
call=rh.price_payoff_fourier(T,ratios,c['rough_model'],c['rough_numerical'],c['market'],{'type':'call'})
put=rh.price_payoff_fourier(T,ratios,c['rough_model'],c['rough_numerical'],c['market'],{'type':'put'})
assert np.max(abs(call-put-disc*F*(1-ratios)))<1e-12
portfolio=rh.price_payoff_fourier(T,ratios,c['rough_model'],c['rough_numerical'],c['market'],{'type':'portfolio','legs':[{'type':'call','weight':2.,'strike_multiplier':1.},{'type':'put','weight':-1.,'strike_multiplier':1.}]})
np.testing.assert_allclose(portfolio,2*call-put,atol=1e-12);record('put_call_parity_and_portfolio_linearity',True)
# Independent analytic deterministic variance case, classical H=.5, xi=0.
model=dict(c['rough_model'],H=.5,xi=0.,v0=.04,theta=.04)
grid={'maturities':[.5],'moneyness':ratios.tolist()}
constant=rh.parameter_greeks_fourier(model,c['rough_numerical'],c['market'],grid,c['rough_greeks'],{'type':'call'})
I=model['theta']*T;d1=np.log(1/ratios)/np.sqrt(I)+.5*np.sqrt(I)
analytic_price=disc*F*(ndtr(d1)-ratios*ndtr(d1-np.sqrt(I)))
assert max(abs(constant[constant.parameter=='H'].discounted_price-analytic_price))<.002
analytic_theta=disc*F*np.exp(-d1*d1/2)/np.sqrt(2*np.pi)/(2*np.sqrt(I))*(T-(1-np.exp(-model['kappa']*T))/model['kappa'])
err=float(max(abs(constant[constant.parameter=='theta'].greek-analytic_theta)))
assert err<.02;record('independent_analytic_theta_greek_max_abs_error',err)
assert max(abs(constant[constant.parameter=='H'].greek))<.01
assert max(abs(constant[constant.parameter=='kappa'].greek))<.01
record('constant_variance_H_and_kappa_zero',True)
assert set(constant[constant.parameter=='H'].stencil)=={'backward_2'}
assert set(constant[constant.parameter=='xi'].stencil)=={'forward_2'}
record('domain_boundary_stencils',True)
# Reject malformed payoff, grid and FD controls.
for bad in [{'steps':{'H':0.,'xi':.001,'kappa':.005,'theta':.0001}},{'fd_rel_tolerance':-1.}]:
 try:rh.parameter_greeks_fourier(c['rough_model'],c['rough_numerical'],c['market'],grid,bad)
 except ValueError:pass
 else:raise AssertionError('Contrôles invalides acceptés')
try:rh.price_payoff_fourier(T,ratios,c['rough_model'],c['rough_numerical'],c['market'],{'type':'arbitrary'})
except ValueError:pass
else:raise AssertionError('Payoff non supporté accepté')
record('invalid_inputs_rejected',True)
# Bump refinement and numerical pricing refinement are separate diagnostics.
subgrid={'maturities':[float(e['grid']['maturities'][0]),float(e['grid']['maturities'][3]),1.],'moneyness':ratios.tolist()}
base=rh.parameter_greeks_fourier(c['rough_model'],c['rough_numerical'],c['market'],subgrid,c['rough_greeks'])
half=copy.deepcopy(c['rough_greeks']);half['steps']={k:v/2 for k,v in half['steps'].items()}
bump=rh.parameter_greeks_fourier(c['rough_model'],c['rough_numerical'],c['market'],subgrid,half)
fine=rh.parameter_greeks_fourier(c['rough_model'],dict(c['rough_numerical'],riccati_steps=384),c['market'],subgrid,c['rough_greeks'])
fourier=rh.parameter_greeks_fourier(c['rough_model'],dict(c['rough_numerical'],fourier_nodes=768,fourier_upper=375.),c['market'],subgrid,c['rough_greeks'])
comparison=base[['T','moneyness','parameter','greek']].copy()
for label,df in [('half_bump',bump),('time_refined',fine),('fourier_refined',fourier)]:
 comparison[label]=df.greek.to_numpy();comparison[label+'_abs_change']=abs(df.greek.to_numpy()-base.greek.to_numpy())
comparison.to_csv(O/'verification_convergence_greeks.csv',index=False)
report['numerical_evidence']['max_abs_changes_by_parameter']=comparison.groupby('parameter')[[x for x in comparison if x.endswith('abs_change')]].max().to_dict()
report['numerical_evidence']['interpretation']='Observed refinement differences; no full convergence guarantee, especially short maturities and wings.'
record('refinement_studies_executed',True)
# Original rough pricing routines unchanged; compare source text before additions.
assert hashlib.sha256((P/'rough_heston.py').read_bytes()[:10252]).hexdigest()=='b038aec13a863fc0ba65572e64c21d5dc8ff87098307ae562c6c651ed06b06fc'
assert hashlib.sha256((P/'mpce.py').read_bytes()).hexdigest()=='fa83dd1d5aef0f16b4d327e86ff265bcc6ea09413c41c461ff95cda2626fe6be'
stored=pd.read_csv(P/'verification_visuelle/surfaces_archivees.csv')
preview=P/'verification_visuelle';preview.mkdir(exist_ok=True)
pairs=vis.create_black_cuts(stored,preview)
for (ax,right),T in zip(pairs,sorted(stored.loc[stored.requested_surface_point,'T'].unique())):
 assert len(ax.lines)==1 and len(right.lines)==1
 assert ax.get_ylabel()=='Volatilité Rough Heston (%)';assert right.get_ylabel()=='Volatilité MPCE (%)'
 for model,axis in [('rough_heston',ax),('mpce',right)]:
  points=stored[(stored.model==model)&stored.requested_surface_point&(stored['T']==T)].sort_values('moneyness')
  np.testing.assert_allclose(axis.lines[0].get_ydata(),100*points.black_iv)
record('mpce_source_byte_identical',True);record('original_rough_pricing_source_unchanged',True);record('dual_axes_match_archived_actual_data',True)
report['elapsed_seconds']=time.perf_counter()-t
report['not_executed']=['MPCE simulation or numerical tests','Full notebook including MPCE execution','Complete convergence over all grid points','Actual Jupyter kernel execution']
(O/'configuration_test.json').write_text(json.dumps(c,indent=2))
(P/'verification_livraison.json').write_text(json.dumps(report,indent=2,ensure_ascii=False))
print(json.dumps(report,indent=2),flush=True)


full_grid_count 528
fd_stable_count 528
greek_plot_points_match_csv True
put_call_parity_and_portfolio_linearity True
independent_analytic_theta_greek_max_abs_error 4.189448321056943e-05
constant_variance_H_and_kappa_zero True
domain_boundary_stencils True
invalid_inputs_rejected True
refinement_studies_executed True
mpce_source_byte_identical True
original_rough_pricing_source_unchanged True
dual_axes_match_archived_actual_data True
{
  "status": "PARTIALLY_VERIFIED",
  "mpce_executed": false,
  "checks": {
    "full_grid_count": 528,
    "fd_stable_count": 528,
    "greek_plot_points_match_csv": true,
    "put_call_parity_and_portfolio_linearity": true,
    "independent_analytic_theta_greek_max_abs_error": 4.189448321056943e-05,
    "constant_variance_H_and_kappa_zero": true,
    "domain_boundary_stencils": true,
    "invalid_inputs_rejected": true,
    "refinement_studies_executed": true,
    "mpce_source_byte_identical": true,
    "original_rough_pricing_source_unchanged": true,
  